In [51]:
import sys, json, warnings, importlib
from pathlib import Path
import pandas as pd, numpy as np

warnings.filterwarnings("ignore")
sys.path.append(str(Path.cwd().parent / "src"))
from config import DATA_PROCESSED, RESULTS_TABLES, RANDOM_SEED

import features.lexical_features as lexical_module
import features.evidentiary_features as evidentiary_module
import features.sentiment_features as sentiment_module
import features.temporal_features as temporal_module

lexical_module = importlib.reload(lexical_module)
evidentiary_module = importlib.reload(evidentiary_module)
sentiment_module = importlib.reload(sentiment_module)
temporal_module = importlib.reload(temporal_module)

add_lexical_features = lexical_module.add_lexical_features
add_evidentiary_features = evidentiary_module.add_evidentiary_features
add_sentiment_features = sentiment_module.add_sentiment_features
add_time_of_day = temporal_module.add_time_of_day

np.random.seed(RANDOM_SEED)
print("Ready")
print("Lexical features reloaded: MTLD and lexical sophistication enabled")

Ready
Lexical features reloaded: MTLD and lexical sophistication enabled


In [52]:
reddit  = pd.read_parquet(DATA_PROCESSED / "reddit_processed.parquet")
twitter = pd.read_parquet(DATA_PROCESSED / "twitter_processed.parquet")
print("Reddit:", reddit.shape, " Twitter:", twitter.shape)

Reddit: (9409, 35)  Twitter: (11733, 27)


In [53]:
reddit  = add_lexical_features(reddit,  "clean_text")
twitter = add_lexical_features(twitter, "clean_text")
print("Lexical done")

Lexical done


In [54]:
# Evidentiary — links need the RAW text, lexicon uses cleaned text
import importlib
import features.evidentiary_features as evidentiary_module

evidentiary_module = importlib.reload(evidentiary_module)
add_evidentiary_features = evidentiary_module.add_evidentiary_features

reddit  = add_evidentiary_features(reddit, raw_col="self_text", text_col="clean_text")
twitter = add_evidentiary_features(twitter, raw_col="tweet_text", text_col="clean_text")

print("Evidentiary done")
print("  Reddit  rows with link_count > 0:", (reddit["link_count"] > 0).sum())
print("  Twitter rows with link_count > 0:", (twitter["link_count"] > 0).sum())
print("  Reddit  mean rigor_score:", round(reddit["rigor_score"].mean(), 4))
print("  Twitter mean rigor_score:", round(twitter["rigor_score"].mean(), 4))

Evidentiary done
  Reddit  rows with link_count > 0: 492
  Twitter rows with link_count > 0: 1
  Reddit  mean rigor_score: 0.031
  Twitter mean rigor_score: 0.0056


In [55]:
#  If your dataset is very large, sample first for testing:
reddit  = reddit.sample(9000, random_state=RANDOM_SEED)
twitter = twitter.sample(9000, random_state=RANDOM_SEED)

import time

started = time.perf_counter()
print(f"Starting Reddit sentiment/emotion features ({len(reddit):,} rows)...", flush=True)
reddit = add_sentiment_features(reddit, "clean_text")
print(f"Reddit complete after {time.perf_counter() - started:.1f}s", flush=True)

print(f"Starting X sentiment/emotion features ({len(twitter):,} rows)...", flush=True)
twitter = add_sentiment_features(twitter, "clean_text")
print(f"X complete after {time.perf_counter() - started:.1f}s", flush=True)
print("Sentiment done")

Starting Reddit sentiment/emotion features (9,000 rows)...
[sentiment] computing on 9,000 rows ...
[sentiment] computing emotions (NRC lexicon) ...
[sentiment] computing polarity (VADER) ...
[sentiment] non-zero scores: 8,194
Reddit complete after 11.4s
Starting X sentiment/emotion features (9,000 rows)...
[sentiment] computing on 9,000 rows ...
[sentiment] computing emotions (NRC lexicon) ...
[sentiment] computing polarity (VADER) ...
[sentiment] non-zero scores: 8,155
X complete after 20.8s
Sentiment done


In [56]:
reddit  = add_time_of_day(reddit)
twitter = add_time_of_day(twitter)
print("Temporal done")

Temporal done


In [57]:
reddit["platform"]  = "Reddit"
twitter["platform"] = "X"

reddit.to_parquet(DATA_PROCESSED / "reddit_with_features.parquet", index=False)
twitter.to_parquet(DATA_PROCESSED / "twitter_with_features.parquet", index=False)

combined = pd.concat([reddit, twitter], ignore_index=True)
combined.to_parquet(DATA_PROCESSED / "combined_with_features.parquet", index=False)

print("Saved. Reddit:", len(reddit), "Twitter:", len(twitter), "Combined:", len(combined))

Saved. Reddit: 9000 Twitter: 9000 Combined: 18000


In [58]:
key = ["fk_grade", "ttr", "rigor_score", "sentiment_score",
       "evidentiary_count", "link_count", "engagement_normalized"]
print("Reddit means:\n",  reddit[key].mean(numeric_only=True).round(4))
print("\nX means:\n",     twitter[key].mean(numeric_only=True).round(4))

Reddit means:
 fk_grade                 8.4658
ttr                      0.8636
rigor_score              0.0311
sentiment_score         -0.1949
evidentiary_count        0.9358
link_count               0.0924
engagement_normalized    0.2251
dtype: float64

X means:
 fk_grade                 10.8042
ttr                       0.8815
rigor_score               0.0056
sentiment_score          -0.2454
evidentiary_count         0.5408
link_count                0.0000
engagement_normalized     0.4184
dtype: float64
